<a href="https://colab.research.google.com/github/harita-gr/ICTACK-AI-ML/blob/main/Intermediate_Assessment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Logistic Regression

Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv("/content/train_LZdllcl.csv")

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

Data Pre-proprocessing

In [ ]:
#Dropping unwanted attributes
data.drop(['region','gender','recruitment_channel'], axis=1, inplace = True)

In [ ]:
# Encode categorical variables using label encoding
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Encode the 'department' column
data['department'] = label_encoder.fit_transform(data['department'])
# Encode the 'education' column
data['education'] = label_encoder.fit_transform(data['education'])

In [ ]:
data.head()

In [ ]:
data['department'].unique()

In [ ]:
data['education'].unique()

Missing Data Handling

1. Replacing missing value of "education" column with mode

In [ ]:
data["education"].nunique()

In [ ]:
# Plot the distribution of education levels
plt.figure(figsize=(5, 4))
sns.countplot(data=data, x='education')
plt.title('Distribution of Education Levels')
plt.xlabel('Education Level')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Calculate the mode of the 'education' column
education_mode = data['education'].mode()[0]

# Replace missing values with the mode
data['education'] = data['education'].fillna(education_mode)

In [ ]:
data.isnull().sum()

2. Using predictive modeling techniques to impute missing values in the 'previous_year_rating' column


In [ ]:
data["previous_year_rating"].nunique()

In [ ]:
# Plot histogram of previous_year_rating
plt.figure(figsize=(8, 6))
sns.histplot(data=data, x='previous_year_rating', bins=10, kde=True)
plt.title('Distribution of Previous Year Ratings')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
# Create a copy of the DataFrame with missing values
data_impute = data.copy()

In [ ]:
# Split the data into training and test sets
train_data = data_impute.dropna()  # Use rows without missing values for training
X_train = train_data.drop(columns=['previous_year_rating'])  # Features
y_train = train_data['previous_year_rating']  # Target variable

In [ ]:
# Prepare data for imputation (rows with missing values)
X_impute = data_impute[data_impute['previous_year_rating'].isna()].drop(columns=['previous_year_rating'])

# Train a simple linear regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

In [ ]:
# Predict missing values
predicted_ratings = linear_model.predict(X_impute)

# Update DataFrame with imputed values
data_impute.loc[data_impute['previous_year_rating'].isna(), 'previous_year_rating'] = predicted_ratings

In [ ]:
data = data_impute
data.isnull().sum()

## Modelling

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression()

In [ ]:
X = data.drop(['is_promoted'], axis=1)  # Features (all columns except 'is_promoted')
y = data['is_promoted']  # Target variable ('is_promoted' column)

In [ ]:
y.head()

In [ ]:
X.head()

In [ ]:
# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
log_model = LogisticRegression()
log_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

# Make predictions on the validation set
y_pred = log_model.predict(X_val)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_val, y_pred))

To predict the output for test file using the trained model

In [ ]:
test_data = pd.read_csv('/content/test_2umaH9m.csv')

In [ ]:
test_data.head()

In [ ]:
test_data.shape

In [ ]:
test_data.isnull().sum()

Pre-Processing Test Data

In [ ]:
#Dropping unwanted attributes
test_data.drop(['region','gender','recruitment_channel'], axis=1, inplace = True)

# Encode the 'department' column
test_data['department'] = label_encoder.fit_transform(test_data['department'])
# Encode the 'education' column
test_data['education'] = label_encoder.fit_transform(test_data['education'])

In [ ]:
test_data.head()

In [ ]:
# Calculate the mode of the 'education' column
education_mode = test_data['education'].mode()[0]

# Replace missing values with the mode
test_data['education'] = test_data['education'].fillna(education_mode)

In [ ]:
# Create a copy of the DataFrame with missing values
data_impute = test_data.copy()

# Split the data into training and test sets
train_data = data_impute.dropna()  # Use rows without missing values for training
X_train = train_data.drop(columns=['previous_year_rating'])  # Features
y_train = train_data['previous_year_rating']  # Target variable

# Prepare data for imputation (rows with missing values)
X_impute = data_impute[data_impute['previous_year_rating'].isna()].drop(columns=['previous_year_rating'])

# Train a simple linear regression model
linear_model_2 = LinearRegression()
linear_model_2.fit(X_train, y_train)

# Predict missing values
predicted_ratings = linear_model_2.predict(X_impute)

# Update DataFrame with imputed values
data_impute.loc[data_impute['previous_year_rating'].isna(), 'previous_year_rating'] = predicted_ratings

test_data = data_impute
test_data.isnull().sum()

Use the trained model to make predictions on the preprocessed test data


In [ ]:
X_test = test_data
predictions = log_model.predict(X_test)

# Create a DataFrame with employee_id and corresponding predictions
predictions_data = pd.DataFrame({'employee_id': test_data['employee_id'], 'is_promoted': predictions})
predictions_data.to_csv('/content/predictions.csv', index=False)